In [27]:
import torch 
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim

# Create the transformer

In [28]:
# this transformer will have the chain's of trasforms that we want to apply on our images
transform=transforms.Compose([
    transforms.ToTensor(), # this fxn will firstly convert the image [0,255] to tensor.then it will apply scaling to it[0,1]. 
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # this fxn will apply normaliztion to it . [-1,1]
    # we have given default values of mean and standard deviation
])

# Import the Dataset

In [29]:
# we have to import the dataset from torchvision.datasets

In [ ]:
from torchvision.datasets import CIFAR10
train_set=CIFAR10(root="./data",train=True,download=True,transform=transform)
test_set=CIFAR10(root="./data",train=False,download=True,transform=transform)

# Convert the data into dataloaders

In [31]:
train_loader=DataLoader(train_set,batch_size=32,shuffle=True)
test_loader=DataLoader(test_set,batch_size=32,shuffle=False)

# Build the CNN Aircheture

In [32]:
class CNN(nn.Module):
    # create the constructor
    def __init__(self):
        super(CNN,self).__init__()
        # create the convo layer
        self.convo_layer=nn.Sequential(
            #1st convo layer
            nn.Conv2d(3,32,kernel_size=3,padding=1,stride=1), # 3=input i.e RGB values, 32=is the filters that we have applied
            # means this layer will give us 32 feature maps . this 32 will be the output of this layer
            nn.ReLU(),
            nn.MaxPool2d(stride=2,kernel_size=2),

            # 2nd convo layer
            nn.Conv2d(32,64,kernel_size=3,padding=1,stride=1), # here 32 are the feature map that has become the input for 
            # this layer 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            #3rd convo layer
            nn.Conv2d(64,128,kernel_size=3,padding=1,stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        # create the fully connected layer
        self.fully_connected_layer=nn.Sequential(
            # the output from the convo layer after flattening 
            # this is the simple ann layer
            nn.Linear(4*4*128,256),
            # the reason why we have taken 4*4*128 is the origional image size was 32 i.e w=32,h=32. after the 
            #first pooling layer the size of image becmae 16 and after the 2rd layer it become  8 and after the 
            # 3rd layer it become 4 i.e width=4 and height=4. 
            # 128 was the feature maps.that we have received from the 3rd convo layer 
            #apply the relu fxn on the sequential layer
            nn.ReLU(),
            nn.Linear(256,10),
        )

    # create the forward fxn
    def forward(self,x):
        # convo layer is applies
        x=self.convo_layer(x)
        # apply the flattening of the feature map's
        x=x.view(x.size(0),-1)
        # apply the fully connected layer
        x=self.fully_connected_layer(x)
        return x

# Create the model

In [33]:
model=CNN()
createrian=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

# train the model 

In [ ]:
epochs=20

print("starting the training outside of epoc")
for epoc in range(epochs):
    
    model.train() #put the model in training mode
    training_loss=0.0

    print("starting the training")
    for images,labels in train_loader:
        # set the gradients to zero
        optimizer.zero_grad()
        output=model.forward(images)
        # calculate the loss
        loss=createrian(output,labels)
        # calculate the loss
        loss.backward()
        # use optimizer to update the weights
        optimizer.step()
        training_loss+=loss.item()
    
    print(f" training loss after the {epoc+1} is {training_loss/len(train_loader)}") 

# Evaluation


In [57]:
model.eval()
correct_pred=0
total_pred=0
with torch.no_grad():
    for images,labels in test_loader:
        output=model.forward(images)
        # the output is a logits vector . we need to find the highest val index of each batch
        max_val,index=torch.max(output,1)
        #here we are comaparing the output's highest index with the labels
        correct_pred+=(index==labels).sum().item()
        total_pred+=len(images)
print("the total pred were ",total_pred)
print("the acc pred are ",correct_pred)

the labesl are  tensor([3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9,
        5, 2, 4, 0, 9, 6, 6, 5])
the labesl are  tensor([4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8,
        7, 7, 4, 6, 7, 3, 6, 3])
the labesl are  tensor([6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7,
        8, 9, 0, 3, 8, 6, 4, 6])
the labesl are  tensor([6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7,
        8, 3, 1, 2, 8, 0, 8, 3])
the labesl are  tensor([5, 2, 4, 1, 8, 9, 1, 2, 9, 7, 2, 9, 6, 5, 6, 3, 8, 7, 6, 2, 5, 2, 8, 9,
        6, 0, 0, 5, 2, 9, 5, 4])
the labesl are  tensor([2, 1, 6, 6, 8, 4, 8, 4, 5, 0, 9, 9, 9, 8, 9, 9, 3, 7, 5, 0, 0, 5, 2, 2,
        3, 8, 6, 3, 4, 0, 5, 8])
the labesl are  tensor([0, 1, 7, 2, 8, 8, 7, 8, 5, 1, 8, 7, 1, 3, 0, 5, 7, 9, 7, 4, 5, 9, 8, 0,
        7, 9, 8, 2, 7, 6, 9, 4])
the labesl are  tensor([3, 9, 6, 4, 7, 6, 5, 1, 5, 8, 8, 0, 4, 0, 5, 5, 1, 1, 8, 9, 0, 3, 1, 9,
 